# AIC 2026 — OCR Demo / Benchmark

Notebook **chỉ để thử nghiệm**: chạy OCR trên một số ít keyframe để trả lời hai câu hỏi:

1. **Tốc độ có ổn không?** — ms/keyframe, tách riêng thời gian detect (CRAFT) và recognize (VietOCR), rồi ngoại suy ra thời gian chạy toàn bộ dataset.
2. **Chất lượng có ổn không?** — vẽ box lên ảnh và in text đọc được để mắt người tự đánh giá.

Tham số OCR (`MIN_CONFIDENCE`, `BOX_PADDING`, …) giữ **y hệt** `AIC_OCR_Keyframes_Colab.ipynb` để số đo có ý nghĩa.

> **Không ghi gì vào Drive.** Kết quả demo lưu ở `/content/ocr_demo/` (bộ nhớ tạm của Colab).
> Dataset mở ở chế độ chỉ đọc.

Trước khi chạy: **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi

# Nguyên tắc: GIỮ NGUYÊN version có sẵn của Colab (torch, numpy, Pillow, opencv…),
# chỉ cài thêm thứ Colab chưa có. Đụng vào chúng là vỡ môi trường.
#
# Hai cái bẫy đã gặp:
#   1. `pip install -U easyocr` — cờ -U upgrade cả DEPENDENCY, kéo torch/numpy/Pillow
#      của Colab lên bản mới. Bỏ -U thì pip thấy dep đã thoả và để nguyên.
#   2. `pip install vietocr` — vietocr 0.3.13 pin `pillow==10.2.0` nên hạ cấp Pillow 11
#      ngay trên cây PIL/ đang dùng, để lại file lẫn version:
#        ImportError: cannot import name 'is_directory' from 'PIL._util'
#      Dùng --no-deps: các dep bị bỏ (pillow, imgaug, albumentations, lmdb,
#      prefetch-generator, scikit-image) chỉ cần khi TRAIN vietocr. Inference chỉ dùng
#      torch / numpy / PIL / yaml / einops / gdown — Colab có sẵn hết trừ einops.
!pip -q install easyocr
!pip -q install --no-deps vietocr
!python -c "import einops" 2>/dev/null || pip -q install --no-deps einops

# Nếu môi trường ĐÃ hỏng từ lần chạy trước, cách sạch nhất là bỏ hẳn session cũ:
# Runtime → Disconnect and delete runtime, rồi mở lại và chạy từ cell này.

In [ ]:
# === CELL SỬA CHỮA — mặc định TẮT, chỉ bật khi đang gặp lỗi ===
#   ImportError: cannot import name 'is_directory' from 'PIL._util'
#
# Nguyên nhân: thư mục PIL/ trên đĩa còn lẫn file của 2 version Pillow khác nhau
# (ImageFont.py bản 9.x + _util.py bản 10/11.x), do lần trước pip hạ cấp Pillow
# ngay trên cây đang dùng. "Restart session" KHÔNG sửa được — file lẫn nằm trên
# ĐĨA, không phải trong RAM.
#
# Đổi thành True rồi chạy cell này. Nó cài lại Pillow nguyên một version nhất quán
# rồi tự restart kernel (bắt buộc, vì PIL đã nạp vào RAM cần được nạp lại).
REPAIR_PILLOW = False   # <-- đổi True khi cần sửa

if REPAIR_PILLOW:
    !pip install -q --force-reinstall --no-cache-dir pillow
    print('Đã cài lại Pillow — kernel restart ngay bây giờ.')
    print('Sau khi restart: đặt lại REPAIR_PILLOW = False, rồi chạy tiếp từ cell mount Drive.')
    import os
    os.kill(os.getpid(), 9)   # buộc restart kernel để nạp lại PIL
else:
    print('Bỏ qua (REPAIR_PILLOW = False).')

# Vẫn lỗi sau khi sửa? => Runtime → Disconnect and delete runtime. Đây là cách chắc chắn nhất
# vì nó xoá sạch đĩa, rồi chạy lại từ cell 1 (đã bỏ -U và dùng --no-deps nên không hỏng nữa).

In [ ]:
# Kiểm tra môi trường — fail sớm ở đây rẻ hơn fail giữa lúc benchmark.
# `import PIL.ImageFont` chính là phép thử: nếu cây PIL/ còn lẫn version thì nó ném
# ImportError ngay tại dòng này.
import PIL, PIL.ImageFont
import easyocr, vietocr, einops, torch

print('Pillow  ', PIL.__version__, '— ImageFont import OK')
print('torch   ', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('einops  ', einops.__version__)
print('\nMôi trường OK.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cấu hình demo

`SAMPLE_VIDEOS` = số video lấy mẫu, `FRAMES_PER_VIDEO` = số keyframe mỗi video. Mặc định 3×20 = 60 ảnh,
chạy khoảng 1–3 phút trên GPU T4. Keyframe được lấy **rải đều** trong video (không lấy 20 frame đầu)
để mẫu phản ánh đúng nội dung trung bình.

In [ ]:
from pathlib import Path

DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
MAP_KEYFRAMES_DIRECTORY = DATASET_DIRECTORY / 'map-keyframes-aic25-b1' / 'map-keyframes'
DEMO_OUTPUT = Path('/content/ocr_demo')   # KHÔNG phải Drive — đây là demo

SAMPLE_VIDEOS = 3
FRAMES_PER_VIDEO = 20

# Giữ giống notebook chính để số đo có ý nghĩa
MIN_CONFIDENCE = 0.35
BOX_PADDING = 4
MIN_BOX_SIDE = 8
RECOGNITION_BATCH = 32
MODEL_ID = 'easyocr-craft-det + vietocr-vgg_transformer'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

DEMO_OUTPUT.mkdir(parents=True, exist_ok=True)
assert DATASET_DIRECTORY.is_dir(), f'Không thấy Dataset_Directory: {DATASET_DIRECTORY}'

keyframe_roots = sorted(p for p in DATASET_DIRECTORY.iterdir()
                        if p.is_dir() and p.name.startswith('Keyframes_'))
assert keyframe_roots, 'Không thấy thư mục Keyframes_* nào đã sync về'
print(f'Thư mục keyframe có sẵn ({len(keyframe_roots)}):', ', '.join(p.name for p in keyframe_roots))

## Chọn mẫu

Đồng thời đếm **tổng số keyframe** của các thư mục đã sync — con số này dùng để ngoại suy tổng thời gian ở cuối.

In [ ]:
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir():
        base = root
    return [p for p in sorted(base.iterdir())
            if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def list_images(video_dir):
    return sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: (not p.stem.isdigit(), int(p.stem) if p.stem.isdigit() else 0, p.name))

all_video_dirs = [v for root in keyframe_roots for v in find_video_dirs(root)]
all_video_dirs.sort(key=lambda p: p.name)
assert all_video_dirs, 'Không tìm thấy video nào'

total_keyframes = sum(len(list_images(v)) for v in all_video_dirs)
print(f'{len(all_video_dirs)} video, tổng {total_keyframes} keyframe (chỉ tính thư mục đã sync)')

# Lấy video rải đều trong danh sách, không lấy 3 video đầu
step = max(1, len(all_video_dirs) // SAMPLE_VIDEOS)
sample_video_dirs = all_video_dirs[::step][:SAMPLE_VIDEOS]

samples = []   # [(video_dir, [image_path, ...])]
for video_dir in sample_video_dirs:
    images = list_images(video_dir)
    if not images:
        continue
    frame_step = max(1, len(images) // FRAMES_PER_VIDEO)
    samples.append((video_dir, images[::frame_step][:FRAMES_PER_VIDEO]))

sample_size = sum(len(imgs) for _, imgs in samples)
for video_dir, images in samples:
    print(f'- {video_dir.name}: lấy {len(images)}/{len(list_images(video_dir))} keyframe')
print('Tổng mẫu:', sample_size, 'ảnh')

## Tải model (đo luôn thời gian load)

In [ ]:
import time
import torch
import easyocr
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

use_gpu = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if use_gpu else 'KHÔNG CÓ — số đo dưới đây sẽ rất chậm và không đại diện')

t0 = time.perf_counter()
detector = easyocr.Reader(['vi'], gpu=use_gpu)
vietocr_config = Cfg.load_config_from_name('vgg_transformer')
vietocr_config['device'] = 'cuda:0' if use_gpu else 'cpu'
vietocr_config['cnn']['pretrained'] = False
recognizer = Predictor(vietocr_config)
load_seconds = time.perf_counter() - t0
print(f'Load model: {load_seconds:.1f}s — {MODEL_ID}')

## OCR có đo thời gian

Cùng logic với notebook chính, chỉ thêm bấm giờ tách 3 pha: **đọc ảnh** (I/O Drive), **detect**, **recognize**.
Tách như vậy để biết nút thắt nằm ở đâu — nếu I/O chiếm phần lớn thì tăng GPU không giúp gì,
phải copy keyframe về đĩa local của Colab trước.

In [ ]:
import numpy as np
from PIL import Image

def detect_boxes(image):
    horizontal, free = detector.detect(np.array(image))
    boxes = []
    for x_min, x_max, y_min, y_max in (horizontal[0] if horizontal else []):
        boxes.append((int(x_min), int(y_min), int(x_max), int(y_max)))
    for polygon in (free[0] if free else []):
        xs = [int(point[0]) for point in polygon]
        ys = [int(point[1]) for point in polygon]
        boxes.append((min(xs), min(ys), max(xs), max(ys)))
    width, height = image.size
    cleaned = []
    for x_min, y_min, x_max, y_max in boxes:
        x_min = max(0, x_min - BOX_PADDING)
        y_min = max(0, y_min - BOX_PADDING)
        x_max = min(width, x_max + BOX_PADDING)
        y_max = min(height, y_max + BOX_PADDING)
        if x_max - x_min >= MIN_BOX_SIDE and y_max - y_min >= MIN_BOX_SIDE:
            cleaned.append((x_min, y_min, x_max, y_max))
    cleaned.sort(key=lambda box: (box[1] // 20, box[0]))
    return cleaned

def recognize_crops(crops):
    results = []
    for start in range(0, len(crops), RECOGNITION_BATCH):
        batch = crops[start:start + RECOGNITION_BATCH]
        try:
            texts, probs = recognizer.predict_batch(batch, return_prob=True)
        except Exception:
            texts, probs = [], []
            for crop in batch:
                text, prob = recognizer.predict(crop, return_prob=True)
                texts.append(text)
                probs.append(prob)
        results.extend(zip(texts, probs))
    return results

def ocr_timed(image_path):
    """OCR một ảnh, trả về (detections_giữ_lại, tất_cả_detections, timing_dict)."""
    t = time.perf_counter()
    image = Image.open(image_path).convert('RGB')
    image.load()
    io_s = time.perf_counter() - t

    t = time.perf_counter()
    boxes = detect_boxes(image)
    detect_s = time.perf_counter() - t

    t = time.perf_counter()
    raw = list(zip(boxes, recognize_crops([image.crop(b) for b in boxes]))) if boxes else []
    recognize_s = time.perf_counter() - t

    everything, kept = [], []
    for box, (text, confidence) in raw:
        item = {'text': (text or '').strip(), 'confidence': float(confidence), 'box': list(box)}
        everything.append(item)
        if item['text'] and item['confidence'] >= MIN_CONFIDENCE:
            kept.append(item)
    timing = {'io': io_s, 'detect': detect_s, 'recognize': recognize_s,
              'total': io_s + detect_s + recognize_s,
              'size': image.size, 'n_boxes': len(boxes)}
    return kept, everything, timing

## Chạy benchmark

Ảnh đầu tiên thường chậm bất thường (cuDNN autotune, cấp phát VRAM) nên được đánh dấu là **warm-up**
và loại khỏi thống kê.

In [ ]:
records = []
run_started = time.perf_counter()

for video_dir, images in samples:
    print(f'\n=== {video_dir.name} ===')
    for image_path in images:
        kept, everything, timing = ocr_timed(image_path)
        records.append({
            'video_id': video_dir.name,
            'keyframe': image_path.name,
            'path': image_path,
            'kept': kept,
            'all': everything,
            'text': ' '.join(d['text'] for d in kept),
            **timing,
        })
        print(f"  {image_path.name}  {timing['total']*1000:6.0f} ms  "
              f"({timing['n_boxes']} box → {len(kept)} giữ lại)  {records[-1]['text'][:70]}")

wall_seconds = time.perf_counter() - run_started
print(f'\nXong {len(records)} ảnh trong {wall_seconds:.1f}s')

## Báo cáo performance

In [ ]:
import statistics

warmup, measured = records[:1], records[1:]
assert measured, 'Cần ít nhất 2 ảnh để bỏ warm-up'

def ms(values):
    return [v * 1000 for v in values]

totals = ms(r['total'] for r in measured)
totals_sorted = sorted(totals)
p50 = statistics.median(totals_sorted)
p90 = totals_sorted[min(len(totals_sorted) - 1, int(0.9 * len(totals_sorted)))]
mean_total = statistics.fmean(totals)

print(f'Mẫu: {len(measured)} ảnh (bỏ {len(warmup)} warm-up: {ms([warmup[0]["total"]])[0]:.0f} ms)')
print(f'Load model một lần: {load_seconds:.1f}s\n')

print('Thời gian / keyframe (ms)')
print(f'  trung bình {mean_total:7.0f}   p50 {p50:7.0f}   p90 {p90:7.0f}   '
      f'min {min(totals):.0f}   max {max(totals):.0f}\n')

print('Chia theo pha (trung bình ms — nút thắt nằm ở pha lớn nhất)')
for phase in ('io', 'detect', 'recognize'):
    avg = statistics.fmean(ms(r[phase] for r in measured))
    print(f'  {phase:10s} {avg:7.0f}  ({avg / mean_total * 100:4.1f}%)')

throughput = 1000 / mean_total
print(f'\nThroughput: {throughput:.2f} keyframe/s  ({throughput * 3600:,.0f} keyframe/giờ)')

eta_hours = total_keyframes * mean_total / 1000 / 3600
print(f'\nNgoại suy toàn bộ {total_keyframes:,} keyframe đã sync:')
print(f'  1 phiên   : {eta_hours:6.1f} giờ')
print(f'  2 phiên //: {eta_hours / 2:6.1f} giờ')
print(f'  4 phiên //: {eta_hours / 4:6.1f} giờ')
print('  (Colab free thường ngắt sau ~4–12h ⇒ cần chạy nhiều phiên; notebook chính đã có cơ chế SKIP để chạy tiếp.)')

if not use_gpu:
    print('\nCẢNH BÁO: đang chạy CPU — bật GPU rồi đo lại, số trên vô nghĩa.')

## Báo cáo chất lượng

Không có ground truth nên **không đo được accuracy**. Các chỉ số dưới đây chỉ là tín hiệu gián tiếp:
tỉ lệ keyframe có chữ, phân bố confidence, và số box bị `MIN_CONFIDENCE` loại — nếu loại quá nhiều
thì ngưỡng đang quá gắt.

In [ ]:
with_text = [r for r in records if r['text']]
all_dets = [d for r in records for d in r['all']]
kept_dets = [d for r in records for d in r['kept']]
dropped = [d for d in all_dets if d['text'] and d['confidence'] < MIN_CONFIDENCE]

print(f"Keyframe có chữ      : {len(with_text)}/{len(records)} ({len(with_text)/len(records)*100:.0f}%)")
print(f"Box detect được      : {len(all_dets)}")
print(f"Box giữ lại (>={MIN_CONFIDENCE}) : {len(kept_dets)}")
print(f"Box bị ngưỡng loại   : {len(dropped)}"
      + (f"  ({len(dropped)/len(all_dets)*100:.0f}% — cân nhắc hạ MIN_CONFIDENCE)" if all_dets and len(dropped)/len(all_dets) > 0.3 else ''))

if kept_dets:
    confs = sorted(d['confidence'] for d in kept_dets)
    print(f"\nConfidence box giữ lại: p10 {confs[len(confs)//10]:.2f}  "
          f"p50 {confs[len(confs)//2]:.2f}  p90 {confs[min(len(confs)-1, len(confs)*9//10)]:.2f}")

print('\n--- Text bị loại vì dưới ngưỡng (kiểm tra xem có mất chữ thật không) ---')
for d in sorted(dropped, key=lambda d: -d['confidence'])[:15]:
    print(f"  {d['confidence']:.2f}  {d['text'][:70]}")
if not dropped:
    print('  (không có)')

## Xem tận mắt — vẽ bounding box lên ảnh

Vẽ box lên `PREVIEW_COUNT` keyframe nhiều chữ nhất — cách nhanh nhất để phát hiện lỗi dấu tiếng Việt,
box cắt cụt chữ, hay logo/watermark bị đọc thành text rác.

Box **xanh lá** = giữ lại (`confidence >= MIN_CONFIDENCE`). Box **đỏ nét mảnh** = bị ngưỡng loại —
bật `SHOW_DROPPED` để xem có đang loại oan chữ thật không.

Ảnh vừa hiện inline vừa được lưu ra `/content/ocr_demo/annotated/*.jpg` để tải về soi kỹ (zoom được,
inline Colab thường quá nhỏ để đọc chữ).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import ImageDraw, ImageFont

PREVIEW_COUNT = 6      # số keyframe muốn xem
SHOW_DROPPED = True    # vẽ luôn box bị MIN_CONFIDENCE loại (màu đỏ)

ANNOTATED_DIR = DEMO_OUTPUT / 'annotated'
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

# DejaVu Sans đi kèm matplotlib, có đủ dấu tiếng Việt — PIL font mặc định thì không.
_font_path = font_manager.findfont('DejaVu Sans')

def annotate(record, show_dropped=SHOW_DROPPED):
    """Vẽ bounding box + text lên ảnh, trả về ảnh PIL mới (không sửa file gốc)."""
    image = Image.open(record['path']).convert('RGB')
    draw = ImageDraw.Draw(image)
    font_size = max(13, image.height // 45)
    font = ImageFont.truetype(_font_path, font_size)
    line_width = max(2, image.height // 400)

    kept_boxes = {tuple(d['box']) for d in record['kept']}
    items = list(record['kept'])
    if show_dropped:
        items += [d for d in record['all']
                  if tuple(d['box']) not in kept_boxes and d['confidence'] < MIN_CONFIDENCE]

    for d in items:
        x_min, y_min, x_max, y_max = d['box']
        is_kept = tuple(d['box']) in kept_boxes
        color = (0, 255, 0) if is_kept else (255, 60, 60)
        draw.rectangle([x_min, y_min, x_max, y_max],
                       outline=color, width=line_width if is_kept else max(1, line_width - 1))

        label = f"{d['text'] or '?'} ({d['confidence']:.2f})"
        left, top, right, bottom = draw.textbbox((0, 0), label, font=font)
        label_w, label_h = right - left, bottom - top
        # Nhãn đặt trên box; nếu sát mép trên thì lật xuống dưới box.
        label_y = y_min - label_h - 3
        if label_y < 0:
            label_y = min(y_max + 2, image.height - label_h - 1)
        label_x = min(x_min, max(0, image.width - label_w - 2))
        draw.rectangle([label_x, label_y, label_x + label_w + 3, label_y + label_h + 3], fill=color)
        draw.text((label_x + 2, label_y + 1), label, fill=(0, 0, 0), font=font)

    return image

top = sorted(with_text, key=lambda r: -len(r['kept']))[:PREVIEW_COUNT]
if not top:
    print('Không keyframe nào có chữ — thử hạ MIN_CONFIDENCE hoặc lấy mẫu video khác.')
else:
    for record in top:
        annotated = annotate(record)
        saved = ANNOTATED_DIR / f"{record['video_id']}_{Path(record['keyframe']).stem}.jpg"
        annotated.save(saved, quality=92)

        n_dropped = len(record['all']) - len(record['kept'])
        plt.figure(figsize=(16, 16 * annotated.height / annotated.width))
        plt.imshow(annotated)
        plt.title(f"{record['video_id']} / {record['keyframe']} — "
                  f"{len(record['kept'])} box giữ lại, {n_dropped} bị loại — "
                  f"{record['total']*1000:.0f} ms", fontsize=11)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        print('Text đọc được:', record['text'] or '(rỗng)')
        print('Đã lưu ảnh:', saved, '\n')

    print(f'{len(top)} ảnh đã vẽ box nằm ở: {ANNOTATED_DIR}')
    print('Tải về máy: mở tab Files (icon thư mục bên trái) → ocr_demo/annotated/ → chuột phải → Download')

## Lưu kết quả demo

Ghi vào `/content/ocr_demo/` (bộ nhớ tạm Colab), **không** đụng tới Drive. Mở file này ra đọc nếu
muốn soi kỹ từng box, hoặc tải về máy để so sánh giữa các lần chỉnh tham số.

In [ ]:
import json

report = {
    'model': MODEL_ID,
    'gpu': torch.cuda.get_device_name(0) if use_gpu else None,
    'min_confidence': MIN_CONFIDENCE,
    'sample_size': len(records),
    'load_seconds': round(load_seconds, 1),
    'ms_per_keyframe': {'mean': round(mean_total, 1), 'p50': round(p50, 1), 'p90': round(p90, 1)},
    'phase_ms': {p: round(statistics.fmean(ms(r[p] for r in measured)), 1)
                 for p in ('io', 'detect', 'recognize')},
    'total_keyframes_synced': total_keyframes,
    'estimated_hours_single_session': round(eta_hours, 1),
    'keyframes_with_text': len(with_text),
    'results': [{'video_id': r['video_id'], 'keyframe': r['keyframe'],
                 'ms': round(r['total'] * 1000), 'text': r['text'], 'detections': r['kept']}
                for r in records],
}
path = DEMO_OUTPUT / 'demo_report.json'
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', path)